In [1]:
"""
CELL 1 — STANDINGS ENGINE
Modular multi-season standings pipeline.
Data source: Opta match-event CSVs in serie_a_*/ folders.

Root-cause fixes applied:
  1. Own goals: goals with 'own goal' == 'Si' are attributed to the OPPOSING
     team (an OG by a home player counts as an away goal, and vice-versa).
  2. Voided / replayed matches: when the same fixture (Home, Away) appears
     more than once in a season, only the LATEST by date is kept
     (the replay is the official result; the voided original is discarded).
  3. 0-0 draws: matches with no type_id == 16 events are valid 0-0 results.
  4. Non-numeric matchday: 'NA' or other prefixes fall back to the CSV
     'week' column.
"""

from __future__ import annotations

import json
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd

# ──────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────
DATA_ROOT = Path("/Users/ricki/Local Projects/FMP_SerieA_Dashboard/data/raw")
SEASONS = ["2021/2022", "2022/2023", "2023/2024", "2024/2025", "2025/2026"]


# ──────────────────────────────────────────────
# STEP 1 — MATCH LOADING
# ──────────────────────────────────────────────
def _parse_match_file(fp: Path, season: str) -> Optional[dict]:
    """Extract match-level row from a single event CSV.

    Own-goal handling:
      A goal event with ``own goal == 'Si'`` was scored by the player listed,
      but the goal is credited to the **opposing** side.
      • home-player OG → away goal
      • away-player OG → home goal

    Returns None only for truly corrupt / empty files.
    """
    try:
        df = pd.read_csv(fp, low_memory=False)

        # ── Filename: {week}_{Home}_{Away}_{matchId}.csv ──
        parts = fp.stem.split("_")
        if len(parts) < 4:
            return None

        raw_week = parts[0]
        home = parts[1]
        away = parts[2]

        # ── Matchday resolution ──
        try:
            week = int(raw_week)
        except ValueError:
            if "week" in df.columns and df["week"].notna().any():
                week = int(df["week"].dropna().iloc[0])
            else:
                week = 0

        # ── Match date (for dedup of voided/replayed matches) ──
        match_date = ""
        if "local_date" in df.columns and df["local_date"].notna().any():
            match_date = str(df["local_date"].dropna().iloc[0])

        # ── Goal counting with own-goal correction ──
        goals = df[df["type_id"] == 16]

        hg = 0
        ag = 0

        if not goals.empty and "team_position" in goals.columns:
            has_og_col = "own goal" in goals.columns
            for _, g in goals.iterrows():
                pos = g["team_position"]
                # Detect own goals
                is_og = False
                if has_og_col:
                    og_val = g["own goal"]
                    is_og = pd.notna(og_val) and str(og_val).strip() == "Si"

                if is_og:
                    # OG: credit the goal to the opposing side
                    if pos == "home":
                        ag += 1
                    else:
                        hg += 1
                else:
                    if pos == "home":
                        hg += 1
                    else:
                        ag += 1

        return {
            "Season": season,
            "Matchday": week,
            "Home": home,
            "Away": away,
            "HG": hg,
            "AG": ag,
            "Date": match_date,
            "File": fp.name,
        }
    except (pd.errors.ParserError, KeyError, ValueError, IndexError):
        return None
    except Exception as exc:
        print(f"  ⚠️  Unexpected error in {fp.name}: {exc}")
        return None


def _dedup_replayed_matches(df: pd.DataFrame) -> pd.DataFrame:
    """Remove voided fixtures when a replay exists.

    If the same (Home, Away) pair appears more than once in a season,
    keep only the row with the latest Date (the official replay).
    """
    if df.empty:
        return df

    fixture_key = df["Home"] + "_" + df["Away"]
    dup_mask = fixture_key.duplicated(keep=False)

    if not dup_mask.any():
        return df

    # Separate clean rows from duplicates
    clean = df[~dup_mask]
    dups = df[dup_mask].copy()

    # Within each duplicate group, keep the latest by date
    kept = dups.sort_values("Date").groupby(["Home", "Away"], as_index=False).last()

    dropped = len(dups) - len(kept)
    if dropped > 0:
        # Show what was removed
        removed = dups.merge(kept, on=list(dups.columns), how="left", indicator=True)
        voided = removed[removed["_merge"] == "left_only"]
        for _, row in voided.iterrows():
            print(f"    🗑️  Voided match dropped: MD{row['Matchday']} "
                  f"{row['Home']} vs {row['Away']} ({row['Date']}) — {row['File']}",
                  flush=True)

    result = pd.concat([clean, kept], ignore_index=True)
    return result.sort_values(["Matchday", "Home"]).reset_index(drop=True)


def load_season_matches(season: str) -> pd.DataFrame:
    """Load all match CSVs from a season folder into a normalized DataFrame."""
    season_folder = DATA_ROOT / f"serie_a_{season.replace('/', '_')}"

    if not season_folder.exists():
        print(f"  ⚠️  {season_folder.name}: folder not found")
        return pd.DataFrame()

    events_folder = season_folder / "events"
    if not events_folder.exists():
        print(f"  ⚠️  {events_folder.name}: events folder not found")
        return pd.DataFrame()

    csv_files = sorted(events_folder.glob("*.csv"))

    records = []
    skipped = 0
    for fp in csv_files:
        row = _parse_match_file(fp, season)
        if row is not None:
            records.append(row)
        else:
            skipped += 1

    df = pd.DataFrame(records)
    if df.empty:
        return df

    # ── Deduplicate voided/replayed matches ──
    before = len(df)
    df = _dedup_replayed_matches(df)
    after = len(df)

    df = df.sort_values(["Matchday", "Home"]).reset_index(drop=True)

    loaded = after
    total = len(csv_files)
    extras = before - after
    parts = [f"📂 {season}: {loaded} matches loaded"]
    if extras > 0:
        parts.append(f"({extras} voided duplicate removed)")
    if skipped > 0:
        parts.append(f"({skipped} skipped — corrupt/unreadable)")
    status = "  " + " ".join(parts)
    print(status + (" ✅" if extras == 0 and skipped == 0 else ""), flush=True)

    return df


# ──────────────────────────────────────────────
# STEP 2 — STANDINGS COMPUTATION
# ──────────────────────────────────────────────
def compute_standings(matches_df: pd.DataFrame) -> pd.DataFrame:
    """Compute league standings per Season from match-level data."""

    def _team_rows(df: pd.DataFrame) -> pd.DataFrame:
        home = df.assign(Team=df["Home"], GF=df["HG"], GA=df["AG"])
        away = df.assign(Team=df["Away"], GF=df["AG"], GA=df["HG"])
        return pd.concat([home, away], ignore_index=True)

    expanded = _team_rows(matches_df)
    expanded["W"] = (expanded["GF"] > expanded["GA"]).astype(int)
    expanded["D"] = (expanded["GF"] == expanded["GA"]).astype(int)
    expanded["L"] = (expanded["GF"] < expanded["GA"]).astype(int)
    expanded["Points"] = expanded["W"] * 3 + expanded["D"]

    standings = (
        expanded
        .groupby(["Season", "Team"], as_index=False)
        .agg(
            MP=("W", "count"),
            W=("W", "sum"),
            D=("D", "sum"),
            L=("L", "sum"),
            GF=("GF", "sum"),
            GA=("GA", "sum"),
            Points=("Points", "sum"),
        )
    )
    standings["GD"] = standings["GF"] - standings["GA"]
    standings = standings[["Season", "Team", "MP", "W", "D", "L",
                           "GF", "GA", "GD", "Points"]]
    standings = (
        standings
        .sort_values(["Season", "Points", "GD", "GF"],
                     ascending=[True, False, False, False])
        .reset_index(drop=True)
    )
    return standings


# ──────────────────────────────────────────────
# STEP 3 — VALIDATION
# ──────────────────────────────────────────────
def validate_season(matches_df: pd.DataFrame, season: str,
                    *, in_progress: bool = False) -> None:
    """Run structural integrity checks on a single season's match data."""
    sdf = matches_df[matches_df["Season"] == season]
    if sdf.empty:
        print(f"  ⚠️  {season}: no data")
        return

    n_matches = len(sdf)
    teams = sorted(set(sdf["Home"].unique()) | set(sdf["Away"].unique()))
    n_teams = len(teams)

    # Match count per team
    home_counts = sdf["Home"].value_counts()
    away_counts = sdf["Away"].value_counts()
    total_counts = home_counts.add(away_counts, fill_value=0).astype(int)

    errors = []

    # Check team count
    if n_teams != 20:
        errors.append(f"Expected 20 teams, found {n_teams}")

    if not in_progress:
        # Completed season checks
        if n_matches != 380:
            errors.append(f"Expected 380 matches, found {n_matches}")
        for team, count in total_counts.items():
            if count != 38:
                errors.append(f"{team} played {count} matches (expected 38)")
    else:
        # In-progress: ensure match count is divisible by 10
        if n_matches % 10 != 0:
            errors.append(f"Match count {n_matches} not divisible by 10")

    # Duplicate fixture check (same Home vs Away more than once)
    fixture_key = sdf["Home"] + " vs " + sdf["Away"]
    dup_fixtures = fixture_key[fixture_key.duplicated()]
    if not dup_fixtures.empty:
        for f in dup_fixtures.unique():
            errors.append(f"Duplicate fixture: {f}")

    if errors:
        print(f"  ❌ {season} validation:")
        for e in errors:
            print(f"       • {e}")
    else:
        print(f"  ✅ {season}: {n_matches} matches, {n_teams} teams — all checks passed")


# ──────────────────────────────────────────────
# STEP 4 — MULTI-SEASON AGGREGATION
# ──────────────────────────────────────────────
print(f"📂 Loading {len(SEASONS)} season(s): {SEASONS}\n")

all_matches_df = pd.concat(
    [load_season_matches(season) for season in SEASONS], ignore_index=True
)
all_standings_df = compute_standings(all_matches_df)

# ── Validation ──
print()
COMPLETED_SEASONS = {"2021/2022", "2022/2023", "2023/2024", "2024/2025"}
for season in SEASONS:
    validate_season(all_matches_df, season,
                    in_progress=(season not in COMPLETED_SEASONS))

# ── Known-result cross-check ──
KNOWN_WINNERS = {
    "2021/2022": ("Milan", 86),
    "2022/2023": ("Napoli", 90),
    "2023/2024": ("Inter", 94),
}
print()
for season, (exp_winner, exp_pts) in KNOWN_WINNERS.items():
    s = all_standings_df[all_standings_df["Season"] == season]
    if s.empty:
        continue
    top = s.iloc[0]
    w_ok = "✅" if top["Team"] == exp_winner else "❌"
    p_ok = "✅" if int(top["Points"]) == exp_pts else f"❌ ({int(top['Points'])})"
    print(f"  {season}: {w_ok} Winner={top['Team']} "
          f"| Pts {p_ok} (expected {exp_pts})")

# ── Summary report ──
print()
for season in all_standings_df["Season"].unique():
    s = all_standings_df[all_standings_df["Season"] == season]
    n_teams = s["Team"].nunique()
    max_md = all_matches_df.loc[all_matches_df["Season"] == season, "Matchday"].max()
    print(f"🏆 {season}  |  {n_teams} teams  |  {max_md} matchdays  |  "
          f"Leader: {s.iloc[0]['Team']} ({int(s.iloc[0]['Points'])} pts)")

print(f"\n✅ all_matches_df  → {all_matches_df.shape}")
print(f"✅ all_standings_df → {all_standings_df.shape}")
all_standings_df.head()

📂 Loading 5 season(s): ['2021/2022', '2022/2023', '2023/2024', '2024/2025', '2025/2026']

  📂 2021/2022: 380 matches loaded ✅
  📂 2021/2022: 380 matches loaded ✅
    🗑️  Voided match dropped: MD25 Spezia vs Verona (2023-03-05) — 25_Spezia_Verona_59b969y6yv9s8m0dsdp8wvwgk.csv
  📂 2022/2023: 380 matches loaded (1 voided duplicate removed)
    🗑️  Voided match dropped: MD25 Spezia vs Verona (2023-03-05) — 25_Spezia_Verona_59b969y6yv9s8m0dsdp8wvwgk.csv
  📂 2022/2023: 380 matches loaded (1 voided duplicate removed)
  📂 2023/2024: 380 matches loaded ✅
  📂 2023/2024: 380 matches loaded ✅
  📂 2024/2025: 380 matches loaded ✅
  📂 2024/2025: 380 matches loaded ✅
  📂 2025/2026: 290 matches loaded ✅
  📂 2025/2026: 290 matches loaded ✅

  ✅ 2021/2022: 380 matches, 20 teams — all checks passed
  ✅ 2022/2023: 380 matches, 20 teams — all checks passed
  ✅ 2023/2024: 380 matches, 20 teams — all checks passed
  ✅ 2024/2025: 380 matches, 20 teams — all checks passed
  ✅ 2025/2026: 290 matches, 20 teams — 

,Season,Team,MP,W,D,L,GF,GA,GD,Points
0,2021/2022,Milan,38,26,8,4,69,31,38,86
1,2021/2022,Inter,38,25,9,4,84,32,52,84
2,2021/2022,Napoli,38,24,7,7,74,31,43,79
3,2021/2022,Juventus,38,20,10,8,57,37,20,70
4,2021/2022,Lazio,38,18,10,10,77,58,19,64


In [2]:
"""
CELL 2 — SEASON POINTS PROGRESSION
Interactive Plotly visualization with dual-mode dropdown.
"""

import plotly.graph_objects as go

# ──────────────────────────────────────────────
# STEP 1 — CUMULATIVE POINTS ENGINE
# ──────────────────────────────────────────────
def compute_points_progression(matches_df: pd.DataFrame) -> pd.DataFrame:
    """Compute per-team cumulative points across matchdays."""

    def _explode_to_team_rows(df: pd.DataFrame) -> pd.DataFrame:
        home = df[["Season", "Matchday", "Home", "HG", "AG"]].copy()
        home.columns = ["Season", "Matchday", "Team", "GF", "GA"]
        away = df[["Season", "Matchday", "Away", "AG", "HG"]].copy()
        away.columns = ["Season", "Matchday", "Team", "GF", "GA"]
        return pd.concat([home, away], ignore_index=True)

    rows = _explode_to_team_rows(matches_df)

    # Match points (3/1/0)
    rows["MatchPoints"] = np.where(
        rows["GF"] > rows["GA"], 3,
        np.where(rows["GF"] == rows["GA"], 1, 0)
    )

    # Sort to guarantee chronological ordering
    rows = rows.sort_values(["Season", "Team", "Matchday"]).reset_index(drop=True)

    # Cumulative sum per (Season, Team)
    rows["CumulativePoints"] = (
        rows.groupby(["Season", "Team"])["MatchPoints"].cumsum()
    )

    return rows[["Season", "Matchday", "Team", "MatchPoints", "CumulativePoints"]]


progression_df = compute_points_progression(all_matches_df)

# ──────────────────────────────────────────────
# STEP 2 — INTERACTIVE VISUALIZATION
# ──────────────────────────────────────────────
SEASONS_LIST = sorted(progression_df["Season"].unique())
TEAMS = sorted(progression_df["Team"].unique())

# ── Colour palette (enough for 20 teams) ──
PALETTE = [
    "#636EFA", "#EF553B", "#00CC96", "#AB63FA", "#FFA15A",
    "#19D3F3", "#FF6692", "#B6E880", "#FF97FF", "#FECB52",
    "#1F77B4", "#FF7F0E", "#2CA02C", "#D62728", "#9467BD",
    "#8C564B", "#E377C2", "#7F7F7F", "#BCBD22", "#17BECF",
]

fig = go.Figure()

# ── MODE A traces (Season View): one trace per (season, team) ──
trace_index = 0
season_button_args: list[dict] = []

for season in SEASONS_LIST:
    sdf = progression_df[progression_df["Season"] == season]
    teams_in_season = sorted(sdf["Team"].unique())
    vis_indices: list[int] = []

    for i, team in enumerate(teams_in_season):
        tdf = sdf[sdf["Team"] == team].sort_values("Matchday")
        fig.add_trace(go.Scatter(
            x=tdf["Matchday"],
            y=tdf["CumulativePoints"],
            mode="lines+markers",
            name=team,
            line=dict(color=PALETTE[i % len(PALETTE)], width=2.2),
            marker=dict(size=4),
            hovertemplate=(
                f"<b>{team}</b><br>"
                "Matchday %{x}<br>"
                "Points: %{y}<extra></extra>"
            ),
            visible=False,
        ))
        vis_indices.append(trace_index)
        trace_index += 1

    season_button_args.append({
        "label": f"Season {season}",
        "vis_indices": vis_indices,
    })

n_mode_a_traces = trace_index

# ── MODE B traces (Team View): one trace per (team, season) ──
team_button_args: list[dict] = []

for team in TEAMS:
    tdf = progression_df[progression_df["Team"] == team]
    seasons_for_team = sorted(tdf["Season"].unique())
    vis_indices = []

    for j, season in enumerate(seasons_for_team):
        sdf = tdf[tdf["Season"] == season].sort_values("Matchday")
        fig.add_trace(go.Scatter(
            x=sdf["Matchday"],
            y=sdf["CumulativePoints"],
            mode="lines+markers",
            name=f"{season}",
            line=dict(color=PALETTE[j % len(PALETTE)], width=2.5),
            marker=dict(size=4),
            hovertemplate=(
                f"<b>{team} — {season}</b><br>"
                "Matchday %{x}<br>"
                "Points: %{y}<extra></extra>"
            ),
            visible=False,
        ))
        vis_indices.append(trace_index)
        trace_index += 1

    team_button_args.append({
        "label": team,
        "vis_indices": vis_indices,
    })

n_total_traces = trace_index

# ── Build dropdown buttons ──
def _make_visibility(active_indices: list[int]) -> list[bool]:
    return [i in active_indices for i in range(n_total_traces)]

# Season buttons
season_buttons = []
for btn in season_button_args:
    season_buttons.append(dict(
        label=btn["label"],
        method="update",
        args=[
            {"visible": _make_visibility(btn["vis_indices"])},
            {"title.text": f"Points Progression — {btn['label']}",
             "xaxis.title.text": "Matchday"},
        ],
    ))

# Team buttons
team_buttons = []
for btn in team_button_args:
    team_buttons.append(dict(
        label=btn["label"],
        method="update",
        args=[
            {"visible": _make_visibility(btn["vis_indices"])},
            {"title.text": f"Points Progression — {btn['label']}",
             "xaxis.title.text": "Matchday"},
        ],
    ))

# ── Set default view: first season ──
default_vis = _make_visibility(season_button_args[0]["vis_indices"])
for i, v in enumerate(default_vis):
    fig.data[i].visible = v

# ── Layout ──
fig.update_layout(
    template="plotly_dark",
    title=dict(
        text=f"Points Progression — Season {SEASONS_LIST[0]}",
        font=dict(size=20),
    ),
    xaxis=dict(
        title="Matchday",
        dtick=1,
        gridcolor="rgba(255,255,255,0.08)",
    ),
    yaxis=dict(
        title="Cumulative Points",
        gridcolor="rgba(255,255,255,0.08)",
    ),
    legend=dict(
        font=dict(size=11),
        bgcolor="rgba(0,0,0,0.4)",
        bordercolor="rgba(255,255,255,0.15)",
        borderwidth=1,
    ),
    updatemenus=[
        # Dropdown 1: Season View
        dict(
            buttons=season_buttons,
            direction="down",
            showactive=True,
            x=0.98,
            xanchor="right",
            y=1.18,
            yanchor="top",
            font=dict(size=12),
            bgcolor="rgba(40,40,60,0.9)",
            bordercolor="rgba(255,255,255,0.2)",
            pad=dict(r=10),
        ),
        # Dropdown 2: Team View
        dict(
            buttons=team_buttons,
            direction="down",
            showactive=True,
            x=0.70,
            xanchor="right",
            y=1.18,
            yanchor="top",
            font=dict(size=12),
            bgcolor="rgba(40,40,60,0.9)",
            bordercolor="rgba(255,255,255,0.2)",
            pad=dict(r=10),
        ),
    ],
    annotations=[
        dict(text="Season View ▾", x=0.98, xref="paper", xanchor="right",
             y=1.22, yref="paper", showarrow=False,
             font=dict(size=13, color="white")),
        dict(text="Team View ▾", x=0.70, xref="paper", xanchor="right",
             y=1.22, yref="paper", showarrow=False,
             font=dict(size=13, color="white")),
    ],
    height=650,
    margin=dict(t=120, l=60, r=30, b=50),
    hovermode="x unified",
)

fig.show()